In [1]:
# Core Imports and Environment Setup
# Automatically installs missing dependencies if needed

import sys
import os
import subprocess
from pathlib import Path
import time


def setup_environment_and_check_dependencies():
    """
    Set up environment, check dependencies, and verify external tools.
    Handles virtual environment creation, package installation, and tool detection.
    
    Returns:
    - bool: True if setup completed successfully, False otherwise
    """
    # --- Environment Report ---
    divider = "=" * 60
    print(f"{divider}\nEnvironment Check\n{divider}")
    print(f"Python        : {sys.version.split()[0]}")
    print(f"Working dir   : {os.getcwd()}")

    # Check for virtual environment
    venv_exists = Path(".venv").exists()
    if not venv_exists:
        print("⚠ Virtual environment (.venv) not found")
        print("   Creating virtual environment and installing dependencies...")
        try:
            # Run install_dependencies.sh
            result = subprocess.run(
                ["bash", "install_dependencies.sh"],
                capture_output=True,
                text=True,
                timeout=600  # 10 minute timeout
            )
            if result.returncode == 0:
                print("✓ Virtual environment created and dependencies installed")
                print("   Please restart the kernel to use the new environment")
            else:
                print(f"⚠ Installation had issues:\n{result.stderr[:500]}")
                print("   Please run manually: ./install_dependencies.sh")
        except FileNotFoundError:
            print("⚠ install_dependencies.sh not found")
            print("   Creating virtual environment manually...")
            subprocess.run([sys.executable, "-m", "venv", ".venv"], check=False)
        except subprocess.TimeoutExpired:
            print("⚠ Installation timed out")
            print("   Please run manually: ./install_dependencies.sh")
        except Exception as e:
            print(f"⚠ Error during installation: {e}")
            print("   Please run manually: ./install_dependencies.sh")

    # Core package check and auto-install
    missing = []
    for pkg in ["Bio", "numpy", "requests"]:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)

    if missing:
        print(f"\n⚠ Missing packages: {', '.join(missing)}")
        print("   Attempting to install missing packages...")
        
        # Determine pip command (use venv pip if available)
        pip_cmd = [sys.executable, "-m", "pip"]
        venv_pip = Path(".venv/bin/pip")
        if venv_pip.exists():
            pip_cmd = [str(venv_pip)]
            print("   Using virtual environment pip...")
        
        # Try to install via pip
        try:
            for pkg in missing:
                if pkg == "Bio":
                    pkg_name = "biopython"
                else:
                    pkg_name = pkg
                
                print(f"   Installing {pkg_name}...")
                result = subprocess.run(
                    pip_cmd + ["install", pkg_name, "--quiet"],
                    capture_output=True,
                    text=True,
                    timeout=300
                )
                if result.returncode == 0:
                    print(f"   ✓ {pkg_name} installed")
                else:
                    print(f"   ⚠ Failed to install {pkg_name}")
                    if result.stderr:
                        print(f"   Error: {result.stderr[:200]}")
        except Exception as e:
            print(f"   ⚠ Error installing packages: {e}")
            print("   Please run: ./install_dependencies.sh")
        
        # Re-check after installation attempt
        still_missing = []
        for pkg in missing:
            try:
                __import__(pkg)
            except ImportError:
                still_missing.append(pkg)
        
        if still_missing:
            print(f"\n⚠ Still missing: {', '.join(still_missing)}")
            print("   Please run: ./install_dependencies.sh")
            print("   Or restart kernel after running: ./install_dependencies.sh")
        else:
            print("\n✓ All core packages now available")
    else:
        print("✓ Core packages loaded")

    # Import core packages (now that they're installed)
    # Declare as global to make them available outside the function
    global np, requests, PDB, PDBIO
    try:
        import numpy as np
        import requests
        from Bio import PDB
        from Bio.PDB import PDBIO
        print("✓ Core modules imported")
    except ImportError as e:
        print(f"⚠ Import error: {e}")
        print("   Please restart kernel and run: ./install_dependencies.sh")
        return False

    # Check for external tools
    print("\n" + divider)
    print("External Tools Check")
    print(divider)

    # Check Rosetta
    rosetta_found = False
    rosetta_bin_path = Path("rosetta/source/bin")
    if rosetta_bin_path.exists():
        # Check for Rosetta binaries (they have .linuxgccrelease extension)
        rosetta_binaries = list(rosetta_bin_path.glob("*.linuxgccrelease"))
        if rosetta_binaries:
            # Check for common Rosetta applications
            common_apps = ["relax", "rosetta_scripts", "docking_protocol", "fixbb"]
            found_apps = []
            for app in common_apps:
                if any(app in str(bin_path) for bin_path in rosetta_binaries):
                    found_apps.append(app)
            
            if found_apps:
                print(f"✓ Rosetta (local: rosetta/source/bin)")
                print(f"   Found applications: {', '.join(found_apps)}")
                rosetta_found = True
            else:
                print("✓ Rosetta binaries found (local: rosetta/source/bin)")
                rosetta_found = True
        else:
            print("⚠ Rosetta source found but binaries not built")
            print("   Build with: ./install_rosetta.sh")
    elif Path("rosetta").exists():
        print("⚠ Rosetta directory found but binaries not built")
        print("   Build with: ./install_rosetta.sh")
    else:
        # Check if Rosetta is in PATH
        if subprocess.run(["which", "rosetta_scripts"], capture_output=True).returncode == 0 or \
           subprocess.run(["which", "relax"], capture_output=True).returncode == 0:
            print("✓ Rosetta (in PATH)")
            rosetta_found = True
        else:
            print("⚠ Rosetta not found (optional, used for structure refinement)")
            print("   Install with: ./install_rosetta.sh")

    print(divider)
    return True

# Call the setup function
setup_environment_and_check_dependencies()

Environment Check
Python        : 3.12.3
Working dir   : \\wsl.localhost\Ubuntu\home\kuhfeldrf\peptide-md-docking

⚠ Missing packages: Bio
   Attempting to install missing packages...
   Using virtual environment pip...
   Installing biopython...
   ⚠ Error installing packages: [WinError 2] The system cannot find the file specified
   Please run: ./install_dependencies.sh

⚠ Still missing: Bio
   Please run: ./install_dependencies.sh
   Or restart kernel after running: ./install_dependencies.sh
⚠ Import error: No module named 'Bio'
   Please restart kernel and run: ./install_dependencies.sh


False

In [7]:
# AlphaFold 4 / ColabFold Peptide Structure Prediction
# Predict peptide 3D structure from amino acid sequence using AlphaFold
# Note: All imports are in Cell 0 above

def predict_peptide_structure_alphafold(sequence, output_dir=".", method="colabfold_api"):
    """
    Predict peptide structure using AlphaFold 4 / ColabFold.
    
    Parameters:
    - sequence: Amino acid sequence (single letter code, e.g., "YPFPGP")
    - output_dir: Directory to save output files
    - method: "colabfold_api" (default) or "colabfold_local" or "alphafold_db"
    
    Returns:
    - Path to predicted PDB file, or None if prediction fails
    """
    sequence = sequence.upper().strip()
    
    # Validate sequence
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    if not all(aa in valid_aa for aa in sequence):
        invalid = [aa for aa in sequence if aa not in valid_aa]
        print(f"⚠ Invalid amino acids in sequence: {set(invalid)}")
        return None
    
    if len(sequence) < 5:
        print("⚠ Sequence too short (minimum 5 amino acids)")
        return None
    
    if len(sequence) > 2000:
        print("⚠ Sequence too long (maximum 2000 amino acids for peptides)")
        return None
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    output_pdb = output_path / f"{sequence}.pdb"
    
    if method == "colabfold_api":
        return _predict_colabfold_api(sequence, output_pdb)
    elif method == "colabfold_local":
        return _predict_colabfold_local(sequence, output_pdb)
    elif method == "alphafold_db":
        return _predict_alphafold_db(sequence, output_pdb)
    else:
        print(f"⚠ Unknown method: {method}")
        return None


def _predict_colabfold_api(sequence, output_pdb):
    """Predict using ESMFold API (free, fast, requires internet)"""
    try:
        import requests
        
        print(f"🔬 Predicting structure for sequence: {sequence}")
        print(f"   Length: {len(sequence)} amino acids")
        print(f"   Using ESMFold API (Meta AI - fast alternative to AlphaFold)...")
        
        # ESMFold API endpoint (free, fast alternative to AlphaFold)
        # Note: For AlphaFold 4 specifically, use ColabFold locally or AlphaFold Server
        api_url = "https://api.esmatlas.com/foldSequence/v1/pdb/"
        
        print("   Submitting sequence to ESMFold API...")
        response = requests.post(api_url, data=sequence, timeout=120)
        
        if response.status_code == 200:
            # Save PDB file
            with open(output_pdb, 'w') as f:
                f.write(response.text)
            print(f"✓ Structure predicted and saved to: {output_pdb}")
            return str(output_pdb)
        else:
            print(f"⚠ API request failed with status {response.status_code}")
            print(f"   Response: {response.text[:200]}")
            print("   Trying alternative method...")
            return _predict_colabfold_local(sequence, output_pdb)
            
    except ImportError:
        print("⚠ requests library not available. Install with: pip install requests")
        return None
    except Exception as e:
        print(f"⚠ Error calling ESMFold API: {e}")
        print("   Trying alternative method...")
        return _predict_colabfold_local(sequence, output_pdb)


def _predict_colabfold_local(sequence, output_pdb):
    """Predict using local ColabFold installation"""
    try:
        import subprocess
        
        print(f"🔬 Predicting structure using local ColabFold...")
        
        # Check if colabfold_batch is available
        colabfold_cmd = "colabfold_batch"
        result = subprocess.run(["which", colabfold_cmd], capture_output=True)
        
        if result.returncode != 0:
            print("⚠ ColabFold not found locally")
            print("   Install with: pip install colabfold")
            print("   Or use method='colabfold_api' for API-based prediction")
            return None
        
        # Create temporary FASTA file
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.fasta', delete=False) as tmp_fasta:
            tmp_fasta.write(f">peptide\n{sequence}\n")
            tmp_fasta_path = tmp_fasta.name
        
        try:
            # Run ColabFold
            cmd = [colabfold_cmd, tmp_fasta_path, str(output_pdb.parent)]
            print(f"   Running: {' '.join(cmd)}")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
            
            if result.returncode == 0:
                # Find the output PDB file
                predicted_files = list(output_pdb.parent.glob(f"*{sequence}*.pdb"))
                if predicted_files:
                    predicted_file = predicted_files[0]
                    if predicted_file != output_pdb:
                        import shutil
                        shutil.copy(predicted_file, output_pdb)
                    print(f"✓ Structure predicted and saved to: {output_pdb}")
                    return str(output_pdb)
                else:
                    print("⚠ Output PDB file not found")
                    return None
            else:
                print(f"⚠ ColabFold failed: {result.stderr}")
                return None
        finally:
            # Clean up temp file
            if os.path.exists(tmp_fasta_path):
                os.unlink(tmp_fasta_path)
                
    except Exception as e:
        print(f"⚠ Error running local ColabFold: {e}")
        return None


def _predict_alphafold_db(sequence, output_pdb):
    """Try to fetch from AlphaFold Database if available"""
    try:
        import requests
        
        print(f"🔬 Searching AlphaFold Database for sequence...")
        
        # AlphaFold Database API
        # Note: This searches for exact matches in the database
        # For custom peptides, use ColabFold instead
        
        # Generate a unique identifier (hash of sequence)
        import hashlib
        seq_hash = hashlib.md5(sequence.encode()).hexdigest()
        
        # Try to fetch from AlphaFold DB (this is a simplified example)
        # In practice, you'd need to use the actual AlphaFold DB API
        print("⚠ AlphaFold Database lookup not fully implemented")
        print("   Use method='colabfold_api' for custom peptide prediction")
        return None
        
    except Exception as e:
        print(f"⚠ Error accessing AlphaFold Database: {e}")
        return None


# predicted_pdb = predict_peptide_structure_alphafold("YPFPGP", method="colabfold_api")
# if predicted_pdb:
#     print(f"Predicted structure saved to: {predicted_pdb}")


# Predict Peptide Structure Using AlphaFold 4

Use AlphaFold 4 / ColabFold to predict 3D structure from peptide sequence


In [8]:
# Alternative: Predict multiple peptides or batch processing

def predict_multiple_peptides(sequences, output_dir="alphafold_predictions"):
    """
    Predict structures for multiple peptide sequences.
    
    Parameters:
    - sequences: List of amino acid sequences
    - output_dir: Directory to save all predictions
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    results = {}
    
    for i, seq in enumerate(sequences, 1):
        print(f"\n[{i}/{len(sequences)}] Processing: {seq}")
        predicted = predict_peptide_structure_alphafold(
            seq,
            output_dir=output_dir,
            method="colabfold_api"
        )
        results[seq] = predicted
        
        # Add a small delay between requests to avoid rate limiting
        if i < len(sequences):
            time.sleep(2)
    
    print("\n" + "=" * 60)
    print("Batch Prediction Summary")
    print("=" * 60)
    for seq, pdb_file in results.items():
        status = "✓" if pdb_file else "✗"
        print(f"{status} {seq}: {pdb_file or 'Failed'}")
    
    return results

# Helper function to extract peptides from FASTA file
def extract_peptides_from_fasta(fasta_file, num_peptides=5):
    """
    Extract peptide sequences from a FASTA file.
    
    Parameters:
    - fasta_file: Path to FASTA file
    - num_peptides: Number of peptides to extract (default: 5)
    
    Returns:
    - List of peptide sequences
    """
    peptides = []
    try:
        with open(fasta_file, 'r') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('>'):
                    peptides.append(line)
                    if len(peptides) >= num_peptides:
                        break
        print(f"✓ Extracted {len(peptides)} peptides from {fasta_file}")
        return peptides
    except FileNotFoundError:
        print(f"⚠ File not found: {fasta_file}")
        return []
    except Exception as e:
        print(f"⚠ Error reading file: {e}")
        return []

# Example: Extract first 5 peptides from intestinal unique peptides.txt
peptide_file = "intestinal unique peptides.txt"
peptide_sequences = extract_peptides_from_fasta(peptide_file, num_peptides=5)

# Display the extracted peptides
print(f"\nExtracted {len(peptide_sequences)} peptides:")
for i, seq in enumerate(peptide_sequences, 1):
    print(f"  {i}. {seq} (length: {len(seq)} amino acids)")

# Uncomment to run batch prediction:
#batch_results = predict_multiple_peptides(peptide_sequences)


✓ Extracted 5 peptides from intestinal unique peptides.txt

Extracted 5 peptides:
  1. EPIPLESREE (length: 10 amino acids)
  2. HLPLPLLQPLMQQVPQPI (length: 18 amino acids)
  3. LLNPTHQIYPVTQPLAPVHNPIS (length: 23 amino acids)
  4. HQIYPVTQPL (length: 10 amino acids)
  5. LAPVHNPI (length: 8 amino acids)


# Rosetta Suite Docking
Using Rosetta for advanced peptide-protein docking and structure refinement

In [9]:
# HPC Cluster Support
# Functions to run Rosetta docking on HPC cluster via SSH and SLURM

import socket
import tempfile
import shutil
from pathlib import Path

# HPC Configuration
HPC_CONFIG = {
    "hostname": "hpc.cqls.oregonstate.edu",  # Main HPC hostname
    "user": os.environ.get("USER", os.environ.get("USERNAME", None)),
    "ssh_key": None,  # Will auto-detect from ~/.ssh/
    "remote_workdir": None,  # Will use $HOME/peptide-md-docking by default
    "module_system": "module",  # or "lmod" depending on HPC
    "rosetta_module": "rosetta",  # Module name for Rosetta on HPC
    "slurm_partition": "short",  # Default partition for jobs
    "slurm_time": "2:00:00",  # Default job time
    "slurm_mem": "8G",  # Default memory
    "slurm_cpus": 4,  # Default CPUs
}

# Detect if running on HPC
def is_on_hpc():
    """Check if code is running on HPC cluster"""
    hostname = socket.gethostname()
    # Check hostname patterns common to HPC systems
    hpc_patterns = ["hpc", "login", "compute", "node", "cqls"]
    return any(pattern in hostname.lower() for pattern in hpc_patterns)

# Set HPC configuration
def configure_hpc(hostname=None, user=None, remote_workdir=None, 
                  rosetta_module=None, slurm_partition=None, slurm_time=None):
    """
    Configure HPC connection settings.
    
    Parameters:
    - hostname: HPC hostname (e.g., "hpc.cqls.oregonstate.edu")
    - user: Username on HPC (default: current user)
    - remote_workdir: Working directory on HPC (default: ~/peptide-md-docking)
    - rosetta_module: Module name for Rosetta (default: "rosetta")
    - slurm_partition: SLURM partition (default: "short")
    - slurm_time: Job time limit (default: "2:00:00")
    """
    global HPC_CONFIG
    
    if hostname:
        HPC_CONFIG["hostname"] = hostname
    if user:
        HPC_CONFIG["user"] = user
    elif not HPC_CONFIG["user"]:
        HPC_CONFIG["user"] = os.environ.get("USER", os.environ.get("USERNAME", "user"))
    if remote_workdir:
        HPC_CONFIG["remote_workdir"] = remote_workdir
    if rosetta_module:
        HPC_CONFIG["rosetta_module"] = rosetta_module
    if slurm_partition:
        HPC_CONFIG["slurm_partition"] = slurm_partition
    if slurm_time:
        HPC_CONFIG["slurm_time"] = slurm_time
    
    print(f"✓ HPC configured:")
    print(f"  Hostname: {HPC_CONFIG['hostname']}")
    print(f"  User: {HPC_CONFIG['user']}")
    print(f"  Remote dir: {HPC_CONFIG['remote_workdir'] or '~/' + Path.cwd().name}")
    print(f"  Rosetta module: {HPC_CONFIG['rosetta_module']}")

# Test SSH connection to HPC
def test_hpc_connection():
    """Test SSH connection to HPC"""
    if is_on_hpc():
        print("✓ Running on HPC cluster directly")
        return True
    
    if not HPC_CONFIG["hostname"]:
        print("⚠ HPC hostname not configured")
        return False
    
    print(f"Testing SSH connection to {HPC_CONFIG['hostname']}...")
    
    try:
        # Build SSH command
        ssh_cmd = ["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=5",
                   f"{HPC_CONFIG['user']}@{HPC_CONFIG['hostname']}", "echo 'connected'"]
        
        result = subprocess.run(ssh_cmd, capture_output=True, text=True, timeout=10)
        
        if result.returncode == 0 and "connected" in result.stdout:
            print(f"✓ SSH connection successful")
            return True
        else:
            print(f"⚠ SSH connection failed:")
            print(f"   {result.stderr}")
            print(f"\nTip: Set up SSH key authentication:")
            print(f"   ssh-keygen -t ed25519 -C 'your_email@example.com'")
            print(f"   ssh-copy-id {HPC_CONFIG['user']}@{HPC_CONFIG['hostname']}")
            return False
    except subprocess.TimeoutExpired:
        print("⚠ SSH connection timed out")
        return False
    except Exception as e:
        print(f"⚠ Error testing SSH connection: {e}")
        return False

# Transfer files to HPC
def transfer_to_hpc(local_path, remote_path=None, recursive=True):
    """
    Transfer file/directory to HPC using SCP.
    
    Parameters:
    - local_path: Local file/directory path
    - remote_path: Remote path (default: same as local in remote workdir)
    - recursive: If True, use -r flag for directories
    
    Returns:
    - Remote path if successful, None otherwise
    """
    if is_on_hpc():
        # Already on HPC, just return path
        return str(Path(local_path).absolute())
    
    local_path = Path(local_path)
    if not local_path.exists():
        print(f"⚠ Local path does not exist: {local_path}")
        return None
    
    # Determine remote path
    if remote_path is None:
        if HPC_CONFIG["remote_workdir"]:
            remote_dir = Path(HPC_CONFIG["remote_workdir"])
        else:
            remote_dir = Path(f"peptide-md-docking")
        remote_path = remote_dir / local_path.name
    
    remote_full = f"{HPC_CONFIG['user']}@{HPC_CONFIG['hostname']}:{remote_path}"
    
    print(f"Transferring {local_path} to {remote_full}...")
    
    try:
        scp_cmd = ["scp", "-o", "BatchMode=yes"]
        if recursive and local_path.is_dir():
            scp_cmd.append("-r")
        scp_cmd.extend([str(local_path), remote_full])
        
        result = subprocess.run(scp_cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode == 0:
            print(f"✓ Transfer successful: {remote_path}")
            return str(remote_path)
        else:
            print(f"⚠ Transfer failed: {result.stderr}")
            return None
    except Exception as e:
        print(f"⚠ Error transferring files: {e}")
        return None

# Transfer files from HPC
def transfer_from_hpc(remote_path, local_path=None):
    """
    Transfer file/directory from HPC using SCP.
    
    Parameters:
    - remote_path: Remote file/directory path
    - local_path: Local path (default: same name in current directory)
    
    Returns:
    - Local path if successful, None otherwise
    """
    if is_on_hpc():
        # Already on HPC, just return path
        remote_p = Path(remote_path)
        if local_path:
            local_p = Path(local_path)
            if remote_p != local_p:
                shutil.copy(remote_p, local_p)
            return str(local_p)
        return str(remote_p.absolute())
    
    if local_path is None:
        local_path = Path(remote_path).name
    
    remote_full = f"{HPC_CONFIG['user']}@{HPC_CONFIG['hostname']}:{remote_path}"
    
    print(f"Transferring {remote_full} to {local_path}...")
    
    try:
        scp_cmd = ["scp", "-o", "BatchMode=yes", "-r", remote_full, str(local_path)]
        result = subprocess.run(scp_cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode == 0:
            print(f"✓ Transfer successful: {local_path}")
            return str(local_path)
        else:
            print(f"⚠ Transfer failed: {result.stderr}")
            return None
    except Exception as e:
        print(f"⚠ Error transferring files: {e}")
        return None

# Run command on HPC via SSH
def run_on_hpc(command, capture_output=True, timeout=3600):
    """
    Run command on HPC via SSH.
    
    Parameters:
    - command: Command string or list of strings
    - capture_output: If True, capture stdout/stderr
    - timeout: Timeout in seconds
    
    Returns:
    - subprocess.CompletedProcess result
    """
    if is_on_hpc():
        # Already on HPC, run locally
        if isinstance(command, str):
            command = command.split()
        return subprocess.run(command, capture_output=capture_output, text=True, timeout=timeout)
    
    if isinstance(command, list):
        command = " ".join(command)
    
    ssh_cmd = ["ssh", "-o", "BatchMode=yes", 
               f"{HPC_CONFIG['user']}@{HPC_CONFIG['hostname']}", command]
    
    return subprocess.run(ssh_cmd, capture_output=capture_output, text=True, timeout=timeout)

# Create SLURM job script
def create_slurm_script(job_name, command, output_file=None, error_file=None,
                       partition=None, time=None, mem=None, cpus=None, modules=None):
    """
    Create SLURM job script.
    
    Parameters:
    - job_name: Job name
    - command: Command to run
    - output_file: SLURM output file (default: {job_name}.out)
    - error_file: SLURM error file (default: {job_name}.err)
    - partition: SLURM partition (default: from HPC_CONFIG)
    - time: Job time limit (default: from HPC_CONFIG)
    - mem: Memory required (default: from HPC_CONFIG)
    - cpus: Number of CPUs (default: from HPC_CONFIG)
    - modules: List of modules to load
    
    Returns:
    - SLURM script content as string
    """
    partition = partition or HPC_CONFIG["slurm_partition"]
    time = time or HPC_CONFIG["slurm_time"]
    mem = mem or HPC_CONFIG["slurm_mem"]
    cpus = cpus or HPC_CONFIG["slurm_cpus"]
    output_file = output_file or f"{job_name}.out"
    error_file = error_file or f"{job_name}.err"
    modules = modules or []
    
    script = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={output_file}
#SBATCH --error={error_file}
#SBATCH --partition={partition}
#SBATCH --time={time}
#SBATCH --mem={mem}
#SBATCH --cpus-per-task={cpus}

# Load modules
"""
    
    if modules:
        for module in modules:
            script += f"module load {module}\n"
    
    script += f"""
# Run command
{command}
"""
    
    return script

# Submit SLURM job
def submit_slurm_job(script_content, remote_script_path=None):
    """
    Submit SLURM job to HPC.
    
    Parameters:
    - script_content: SLURM script content (string)
    - remote_script_path: Path to save script on HPC (default: auto-generated)
    
    Returns:
    - Job ID if successful, None otherwise
    """
    if is_on_hpc():
        # Save script locally and submit
        if remote_script_path is None:
            remote_script_path = f"job_{int(time.time())}.sh"
        
        with open(remote_script_path, 'w') as f:
            f.write(script_content)
        
        os.chmod(remote_script_path, 0o755)
        
        result = subprocess.run(["sbatch", remote_script_path], 
                               capture_output=True, text=True)
        
        if result.returncode == 0:
            # Extract job ID from output like "Submitted batch job 12345"
            job_id = result.stdout.strip().split()[-1]
            print(f"✓ Job submitted: {job_id}")
            return job_id
        else:
            print(f"⚠ Job submission failed: {result.stderr}")
            return None
    else:
        # Transfer script to HPC and submit
        # Create temporary local script
        with tempfile.NamedTemporaryFile(mode='w', suffix='.sh', delete=False) as tmp:
            tmp.write(script_content)
            tmp_path = tmp.name
        
        try:
            # Transfer to HPC
            if remote_script_path is None:
                remote_script_path = f"job_{int(time.time())}.sh"
            
            remote_full = transfer_to_hpc(tmp_path, remote_script_path)
            
            if not remote_full:
                return None
            
            # Submit job via SSH
            submit_cmd = f"cd {Path(remote_full).parent} && chmod +x {Path(remote_full).name} && sbatch {Path(remote_full).name}"
            result = run_on_hpc(submit_cmd, timeout=30)
            
            if result.returncode == 0:
                job_id = result.stdout.strip().split()[-1]
                print(f"✓ Job submitted: {job_id}")
                return job_id
            else:
                print(f"⚠ Job submission failed: {result.stderr}")
                return None
        finally:
            # Clean up temp file
            if os.path.exists(tmp_path):
                os.unlink(tmp_path)

# Check job status
def check_job_status(job_id):
    """Check status of SLURM job"""
    if is_on_hpc():
        result = subprocess.run(["squeue", "-j", str(job_id)], 
                               capture_output=True, text=True)
    else:
        result = run_on_hpc(f"squeue -j {job_id}", timeout=10)
    
    if result.returncode == 0:
        lines = result.stdout.strip().split('\n')
        if len(lines) > 1:
            # Job is in queue or running
            return result.stdout
        else:
            # Job completed or not found
            return "COMPLETED"
    return None

# Wait for job completion
def wait_for_job(job_id, check_interval=30, max_wait=None):
    """
    Wait for SLURM job to complete.
    
    Parameters:
    - job_id: Job ID
    - check_interval: Seconds between status checks
    - max_wait: Maximum wait time in seconds (None for no limit)
    
    Returns:
    - True if job completed, False if timeout
    """
    start_time = time.time()
    
    while True:
        status = check_job_status(job_id)
        
        if status == "COMPLETED":
            print(f"✓ Job {job_id} completed")
            return True
        
        if max_wait and (time.time() - start_time) > max_wait:
            print(f"⚠ Wait timeout for job {job_id}")
            return False
        
        time.sleep(check_interval)
        print(f"  Job {job_id} still running...")

print("=" * 60)
print("HPC Support Functions")
print("=" * 60)

# Configure HPC (user can customize)
# Default assumes hpc.cqls.oregonstate.edu
configure_hpc()

# Test connection
HPC_ENABLED = test_hpc_connection()

if HPC_ENABLED:
    print("\n✓ HPC support ready")
    if not is_on_hpc():
        print("  Running locally, will submit jobs to HPC via SSH")
    else:
        print("  Running directly on HPC cluster")
else:
    print("\n⚠ HPC connection not available")
    print("  Will run jobs locally")

print("=" * 60)


HPC Support Functions
✓ HPC configured:
  Hostname: hpc.cqls.oregonstate.edu
  User: kuhfeldr
  Remote dir: ~/peptide-md-docking
  Rosetta module: rosetta
Testing SSH connection to hpc.cqls.oregonstate.edu...
⚠ SSH connection failed:
   
You Are Accessing an Oregon State University System
         Unauthorized Access Prohibited
     Use Constitutes a Consent to Monitoring
       Users have No Expectation of Privacy

kuhfeldr@hpc.cqls.oregonstate.edu: Permission denied (publickey,password,keyboard-interactive).


Tip: Set up SSH key authentication:
   ssh-keygen -t ed25519 -C 'your_email@example.com'
   ssh-copy-id kuhfeldr@hpc.cqls.oregonstate.edu

⚠ HPC connection not available
  Will run jobs locally


In [10]:
# Rosetta Suite Integration
# Comprehensive Rosetta tool detection and integration


def _find_rosetta_binaries():
    """
    Find Rosetta binaries from multiple possible locations:
    1. System PATH
    2. Local rosetta/source/bin directory (from install_rosetta.sh)
    3. Conda environment
    """
    rosetta_bin = {}
    rosetta_paths = []
    
    # Check system PATH
    for cmd in ["rosetta_scripts", "relax", "docking_protocol", "fixbb"]:
        result = subprocess.run(["which", cmd], capture_output=True)
        if result.returncode == 0:
            rosetta_bin[cmd] = result.stdout.decode().strip()
            rosetta_paths.append(Path(rosetta_bin[cmd]).parent)
    
    # Check local rosetta installation (from install_rosetta.sh)
    # Rosetta structure: rosetta/source/bin (binaries built here after ./install_rosetta.sh)
    rosetta_dir = Path("rosetta")
    local_rosetta_bin = Path("rosetta/source/bin")
    
    # Verify rosetta directory exists
    if rosetta_dir.exists() and rosetta_dir.is_dir():
        # Check if binaries directory exists
        if local_rosetta_bin.exists() and local_rosetta_bin.is_dir():
            # Look for actual Rosetta binaries (not just any executable)
            rosetta_binary_names = ["relax", "rosetta_scripts", "docking_protocol", "fixbb", 
                                    "rosetta_scripts.linuxgccrelease", "relax.linuxgccrelease"]
            for cmd_file in local_rosetta_bin.glob("*"):
                if cmd_file.is_file() and os.access(cmd_file, os.X_OK):
                    cmd_name = cmd_file.name
                    # Check if it's a Rosetta binary (by name or if directory has Rosetta binaries)
                    if cmd_name in rosetta_binary_names or any(rosetta_name in cmd_name for rosetta_name in ["rosetta", "relax", "docking", "fixbb"]):
                        if cmd_name not in rosetta_bin:
                            rosetta_bin[cmd_name] = str(cmd_file.absolute())
                            rosetta_paths.append(local_rosetta_bin)
                            
                            # Also add simplified key (without extension) for easier lookup
                            # e.g., "relax.linuxgccrelease" -> also add "relax"
                            if ".linuxgccrelease" in cmd_name:
                                base_name = cmd_name.replace(".linuxgccrelease", "")
                                # Remove .default if present
                                if base_name.endswith(".default"):
                                    base_name = base_name.replace(".default", "")
                                # Only add if not already present (prefer non-extension version)
                                if base_name not in rosetta_bin:
                                    rosetta_bin[base_name] = str(cmd_file.absolute())
    
    # Check conda environment
    if "CONDA_PREFIX" in os.environ:
        conda_bin = Path(os.environ["CONDA_PREFIX"]) / "bin"
        if conda_bin.exists():
            for cmd in ["rosetta_scripts", "relax"]:
                conda_cmd = conda_bin / cmd
                if conda_cmd.exists() and cmd not in rosetta_bin:
                    rosetta_bin[cmd] = str(conda_cmd)
                    rosetta_paths.append(conda_bin)
    
    return rosetta_bin, rosetta_paths

def get_rosetta_command(cmd_name):
    """Get full path to a Rosetta command"""
    # First try exact match
    if cmd_name in rosetta_commands:
        return rosetta_commands[cmd_name]
    
    # Try with .linuxgccrelease extension
    if f"{cmd_name}.linuxgccrelease" in rosetta_commands:
        return rosetta_commands[f"{cmd_name}.linuxgccrelease"]
    
    # Try with .default.linuxgccrelease extension
    if f"{cmd_name}.default.linuxgccrelease" in rosetta_commands:
        return rosetta_commands[f"{cmd_name}.default.linuxgccrelease"]
    
    # Try to find any key that starts with cmd_name
    for key in rosetta_commands.keys():
        if key.startswith(cmd_name):
            return rosetta_commands[key]
    
    return None

# Detect Rosetta installation
rosetta_commands, rosetta_paths = _find_rosetta_binaries()
ROSETTA_AVAILABLE = len(rosetta_commands) > 0

# Print status
print("=" * 60)
print("Rosetta Suite Detection")
print("=" * 60)

if ROSETTA_AVAILABLE:
    print(f"✓ Rosetta found ({len(rosetta_commands)} command(s))")
    if rosetta_paths:
        unique_paths = list(set(str(p) for p in rosetta_paths))
        print(f"  Location(s): {', '.join(unique_paths[:2])}")
    
    # Show only relevant tools for docking workflow
    relevant_tools = ["relax", "rosetta_scripts", "docking_protocol", "fixbb"]
    found_relevant = []
    
    print("\nRelevant Rosetta tools for docking workflow:")
    for tool in relevant_tools:
        if tool in rosetta_commands:
            print(f"  ✓ {tool}")
            found_relevant.append(tool)
        else:
            # Check for variants with extensions
            variants = [k for k in rosetta_commands.keys() if k.startswith(tool) and tool in k]
            if variants:
                # Prefer the one without extension, or first variant
                preferred = [v for v in variants if not v.endswith('.linuxgccrelease')]
                if preferred:
                    print(f"  ✓ {preferred[0]}")
                    found_relevant.append(tool)
                else:
                    print(f"  ✓ {variants[0]}")
                    found_relevant.append(tool)
    
    if len(found_relevant) < len(relevant_tools):
        missing = [t for t in relevant_tools if t not in found_relevant]
        if missing:
            print(f"\n  ⚠ Note: Some tools not found: {', '.join(missing)}")
    
    # Set ROSETTA_BIN_PATH for subprocess calls
    if rosetta_paths:
        ROSETTA_BIN_PATH = str(rosetta_paths[0])
        os.environ["ROSETTA_BIN_PATH"] = ROSETTA_BIN_PATH
    else:
        ROSETTA_BIN_PATH = None
else:
    print("⚠ Rosetta not found")
    print("\nInstallation options:")
    print("  1. From GitHub: ./install_rosetta.sh")
    print("  2. Via conda: conda install -c conda-forge rosetta")
    print("  3. Manual: See INSTALL_MANUAL.md")
    print("\nNote: Rosetta is required for this workflow")
    ROSETTA_BIN_PATH = None

print("=" * 60)


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [ ]:
# Rosetta Peptide-Protein Docking
# Use Rosetta for advanced peptide docking against receptor

def rosetta_peptide_docking(receptor_pdb, peptide_pdb, output_dir="rosetta_docking",
                            nstruct=10, relax=True, use_hpc=None, wait_for_completion=True):
    """
    Perform peptide-protein docking using Rosetta.
    
    Parameters:
    - receptor_pdb: Path to receptor PDB file
    - peptide_pdb: Path to peptide PDB file
    - output_dir: Directory to save results
    - nstruct: Number of structures to generate (default: 10)
    - relax: Whether to relax structures after docking (default: True)
    - use_hpc: If True, run on HPC; if False, run locally; if None, auto-detect
    - wait_for_completion: If True, wait for HPC job to complete (default: True)
    
    Returns:
    - Path to best docked structure, or None if docking fails
    - If HPC job submitted and wait_for_completion=False, returns job_id
    """
    # Determine if we should use HPC
    if use_hpc is None:
        use_hpc = HPC_ENABLED and not is_on_hpc()
    
    # If using HPC and not on HPC, submit job and transfer files
    if use_hpc and not is_on_hpc():
        return _rosetta_peptide_docking_hpc(receptor_pdb, peptide_pdb, output_dir,
                                           nstruct, relax, wait_for_completion)
    
    # Local execution (on HPC or local machine)
    if not is_on_hpc() and not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available locally. Consider using HPC:")
        print("   rosetta_peptide_docking(..., use_hpc=True)")
        return None
    
    # If on HPC but Rosetta not in PATH, try loading module
    if is_on_hpc():
        # Try to find Rosetta or use module system
        rosetta_check = subprocess.run(["which", "rosetta_scripts"], capture_output=True)
        if rosetta_check.returncode != 0:
            # Try loading module
            module_result = subprocess.run(
                f"module load {HPC_CONFIG['rosetta_module']} && which rosetta_scripts",
                shell=True, capture_output=True, text=True
            )
            if module_result.returncode == 0:
                print(f"✓ Loaded Rosetta module: {HPC_CONFIG['rosetta_module']}")
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Get Rosetta command using helper function
    rosetta_scripts = get_rosetta_command("rosetta_scripts")
    if not rosetta_scripts:
        # Try direct command if not found
        rosetta_scripts = "rosetta_scripts"
        result = subprocess.run(["which", rosetta_scripts], capture_output=True)
        if result.returncode != 0:
            print("⚠ rosetta_scripts not found")
            if HPC_ENABLED:
                print("   Try: rosetta_peptide_docking(..., use_hpc=True)")
            return None
    
    print("=" * 60)
    print("Rosetta Peptide-Protein Docking")
    if is_on_hpc():
        print("(Running on HPC cluster)")
    print("=" * 60)
    print(f"Receptor: {receptor_pdb}")
    print(f"Peptide:  {peptide_pdb}")
    print(f"Output:   {output_dir}")
    print(f"Structures: {nstruct}")
    
    # Create Rosetta XML script for peptide docking
    xml_content = """<?xml version="1.0"?>
<ROSETTASCRIPTS>
    <SCOREFXNS>
        <ScoreFunction name="ref2015" weights="ref2015"/>
        <ScoreFunction name="ref2015_cart" weights="ref2015_cart"/>
    </SCOREFXNS>
    
    <MOVERS>
        <Docking name="docking" fullatom="1" local_refine="1" score_high="ref2015"/>
        <FastRelax name="relax" scorefxn="ref2015_cart" />
    </MOVERS>
    
    <PROTOCOLS>
        <Add mover_name="docking"/>
        <Add mover_name="relax"/>
    </PROTOCOLS>
</ROSETTASCRIPTS>
"""
    
    # Write XML to file for Rosetta
    xml_script = output_path / "peptide_docking.xml"
    with open(xml_script, "w") as f:
        f.write(xml_content)
    
    # Prepare input files
    combined_pdb = output_path / "input_complex.pdb"
    receptor_chains, peptide_chain = _combine_pdb_files(receptor_pdb, peptide_pdb, combined_pdb)
    
    if not receptor_chains or not peptide_chain:
        print("⚠ Error: Could not determine chain IDs for docking partners")
        return None
    
    # Format partners string: receptor_chains_peptide_chain (e.g., "A_B" or "ABC_D")
    partners_str = f"{''.join(receptor_chains)}_{peptide_chain}"
    print(f"Docking partners: {partners_str}")
    
    # Run Rosetta docking
    try:
        cmd = [
            rosetta_scripts,
            "-parser:protocol", str(xml_script),
            "-s", str(combined_pdb),
            "-partners", partners_str,
            "-nstruct", str(nstruct),
            "-out:path:pdb", str(output_path),
            "-out:file:scorefile", str(output_path / "docking_scores.sc"),
            "-ex1", "-ex2",
            "-use_input_sc",
            "-flip_HNQ",
            "-no_optH", "false"
        ]
        
        if relax:
            cmd.extend(["-relax:fast"])
        
        print(f"\nRunning: {' '.join(cmd[:5])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            # Find best structure based on score
            best_structure = _find_best_rosetta_structure(output_path)
            if best_structure:
                print(f"\n✓ Docking completed successfully!")
                print(f"Best structure: {best_structure}")
                return str(best_structure)
            else:
                print("\n⚠ Docking completed but best structure not found")
                return None
        else:
            print(f"\n⚠ Rosetta docking failed:")
            print(result.stderr[:500])
            return None
            
    except Exception as e:
        print(f"\n⚠ Error running Rosetta: {e}")
        return None


def _rosetta_peptide_docking_hpc(receptor_pdb, peptide_pdb, output_dir,
                                 nstruct=10, relax=True, wait_for_completion=True):
    """
    Submit Rosetta docking job to HPC via SLURM.
    """
    print("=" * 60)
    print("Rosetta Peptide-Protein Docking (HPC)")
    print("=" * 60)
    print(f"Receptor: {receptor_pdb}")
    print(f"Peptide:  {peptide_pdb}")
    print(f"Output:   {output_dir}")
    print(f"Structures: {nstruct}")
    
    # Determine remote work directory
    if HPC_CONFIG["remote_workdir"]:
        remote_workdir = Path(HPC_CONFIG["remote_workdir"])
    else:
        remote_workdir = Path(f"peptide-md-docking")
    
    # Transfer input files to HPC
    print("\nTransferring files to HPC...")
    remote_receptor = transfer_to_hpc(receptor_pdb, remote_workdir / Path(receptor_pdb).name)
    remote_peptide = transfer_to_hpc(peptide_pdb, remote_workdir / Path(peptide_pdb).name)
    
    if not remote_receptor or not remote_peptide:
        print("⚠ Failed to transfer input files")
        return None
    
    # Create remote output directory
    remote_output = remote_workdir / output_dir
    run_on_hpc(f"mkdir -p {remote_output}", timeout=10)
    
    # Create XML script content (will be written on HPC)
    xml_content = """<?xml version="1.0"?>
<ROSETTASCRIPTS>
    <SCOREFXNS>
        <ScoreFunction name="ref2015" weights="ref2015"/>
        <ScoreFunction name="ref2015_cart" weights="ref2015_cart"/>
    </SCOREFXNS>
    
    <MOVERS>
        <Docking name="docking" fullatom="1" local_refine="1" score_high="ref2015"/>
        <FastRelax name="relax" scorefxn="ref2015_cart" />
    </MOVERS>
    
    <PROTOCOLS>
        <Add mover_name="docking"/>
        <Add mover_name="relax"/>
    </PROTOCOLS>
</ROSETTASCRIPTS>
"""
    
    # Create a Python script to combine PDBs (will be transferred to HPC)
    combine_script_content = f"""#!/usr/bin/env python3
from Bio import PDB
from pathlib import Path
import sys

receptor_pdb = "{remote_receptor}"
peptide_pdb = "{remote_peptide}"
combined_pdb = "{remote_output}/input_complex.pdb"

parser = PDB.PDBParser(QUIET=True)
receptor_structure = parser.get_structure("receptor", receptor_pdb)
peptide_structure = parser.get_structure("peptide", peptide_pdb)

receptor_model = list(receptor_structure.get_models())[0]
peptide_model = list(peptide_structure.get_models())[0]
receptor_chains = [chain.id for chain in receptor_model]
new_chain_id = chr(ord('A') + len(receptor_chains))

new_chain = PDB.Chain.Chain(new_chain_id)
for chain in peptide_model:
    for residue in chain:
        new_chain.add(residue.copy())

receptor_model.add(new_chain)

io = PDB.PDBIO()
io.set_structure(receptor_structure)
io.save(combined_pdb)

partners_str = "{{}}{{}}_{{}}".format("".join(receptor_chains), new_chain_id) if receptor_chains else "A_B"
print(partners_str)
"""
    
    # Transfer combine script to HPC
    combine_script_path = f"{remote_output}/combine_pdbs.py"
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as tmp:
        tmp.write(combine_script_content)
        tmp_path = tmp.name
    
    try:
        combine_script_remote = transfer_to_hpc(tmp_path, combine_script_path)
        if not combine_script_remote:
            print("⚠ Failed to transfer combine script")
            return None
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
    
    # Make script executable
    run_on_hpc(f"chmod +x {combine_script_path}", timeout=10)
    
    # Create wrapper script that combines PDBs and runs Rosetta
    relax_flag = "-relax:fast" if relax else ""
    wrapper_script = f"""#!/bin/bash
cd {remote_workdir}

# Load Rosetta module
module load {HPC_CONFIG['rosetta_module']} 2>/dev/null || true

# Create output directory
mkdir -p {remote_output}

# Combine PDB files and get partners string
PARTNERS=$(python3 {combine_script_path})

# Write XML script
cat > {remote_output}/peptide_docking.xml << 'XMLEOF'
{xml_content}
XMLEOF

# Run Rosetta
rosetta_scripts -parser:protocol {remote_output}/peptide_docking.xml \\
    -s {remote_output}/input_complex.pdb \\
    -partners $PARTNERS \\
    -nstruct {nstruct} \\
    -out:path:pdb {remote_output} \\
    -out:file:scorefile {remote_output}/docking_scores.sc \\
    -ex1 -ex2 -use_input_sc -flip_HNQ -no_optH false {relax_flag}

echo "Docking completed"
"""
    
    # Submit SLURM job
    job_name = f"rosetta_dock_{Path(output_dir).name}"
    script_content = create_slurm_script(
        job_name=job_name,
        command=wrapper_script,
        output_file=f"{remote_output}/{job_name}.out",
        error_file=f"{remote_output}/{job_name}.err",
        modules=[HPC_CONFIG["rosetta_module"]]
    )
    
    job_id = submit_slurm_job(script_content, f"{remote_output}/{job_name}.sh")
    
    if not job_id:
        return None
    
    if not wait_for_completion:
        print(f"\n✓ Job {job_id} submitted to HPC")
        print(f"  Check status: check_job_status({job_id})")
        print(f"  Results will be in: {remote_output}")
        return job_id
    
    # Wait for job completion
    print(f"\nWaiting for job {job_id} to complete...")
    if wait_for_job(job_id):
        # Transfer results back
        print("\nTransferring results from HPC...")
        local_output = Path(output_dir)
        local_output.mkdir(parents=True, exist_ok=True)
        
        # Transfer all output files
        transfer_from_hpc(f"{remote_output}/*", str(local_output))
        
        # Find best structure
        best_structure = _find_best_rosetta_structure(local_output)
        if best_structure:
            print(f"\n✓ Docking completed successfully!")
            print(f"Best structure: {best_structure}")
            return str(best_structure)
        else:
            print("\n✓ Job completed. Check output directory for results.")
            return str(local_output)
    else:
        print(f"\n⚠ Job {job_id} did not complete within timeout")
        print(f"  Check status manually: squeue -j {job_id}")
        print(f"  Results will be in: {remote_output}")
        return None


def _combine_pdb_files(receptor_pdb, peptide_pdb, output_pdb):
    """
    Combine receptor and peptide PDB files.
    
    Returns:
    - tuple: (receptor_chain_ids, peptide_chain_id) or (None, None) on error
    """
    try:
        from Bio import PDB
        
        parser = PDB.PDBParser(QUIET=True)
        receptor_structure = parser.get_structure("receptor", receptor_pdb)
        peptide_structure = parser.get_structure("peptide", peptide_pdb)
        
        # Add peptide as a new chain to receptor
        receptor_model = list(receptor_structure.get_models())[0]
        peptide_model = list(peptide_structure.get_models())[0]
        
        # Get receptor chain IDs
        receptor_chains = [chain.id for chain in receptor_model]
        
        # Get next chain ID for peptide
        new_chain_id = chr(ord('A') + len(receptor_chains))
        
        # Create new chain for peptide
        new_chain = PDB.Chain.Chain(new_chain_id)
        for chain in peptide_model:
            for residue in chain:
                new_chain.add(residue.copy())
        
        receptor_model.add(new_chain)
        
        # Save combined structure
        io = PDB.PDBIO()
        io.set_structure(receptor_structure)
        io.save(str(output_pdb))
        
        return (receptor_chains, new_chain_id)
    except Exception as e:
        print(f"⚠ Error combining PDB files: {e}")
        return (None, None)


def _find_best_rosetta_structure(output_path):
    """Find best structure from Rosetta docking results"""
    score_file = output_path / "docking_scores.sc"
    
    if not score_file.exists():
        # Look for any PDB files
        pdb_files = list(output_path.glob("*.pdb"))
        if pdb_files:
            return pdb_files[0]
        return None
    
    try:
        best_score = float('inf')
        best_structure = None
        
        with open(score_file, 'r') as f:
            for line in f:
                if line.startswith("SCORE:"):
                    parts = line.split()
                    if "total_score" in parts:
                        score_idx = parts.index("total_score")
                        if score_idx + 1 < len(parts):
                            try:
                                score = float(parts[score_idx + 1])
                                if score < best_score:
                                    best_score = score
                                    # Find corresponding PDB file
                                    if "description" in parts:
                                        desc_idx = parts.index("description")
                                        if desc_idx + 1 < len(parts):
                                            desc = parts[desc_idx + 1]
                                            pdb_file = output_path / f"{desc}.pdb"
                                            if pdb_file.exists():
                                                best_structure = pdb_file
                            except (ValueError, IndexError):
                                continue
        
        return best_structure
        
    except Exception as e:
        print(f"⚠ Error parsing score file: {e}")
        # Fallback: return first PDB file
        pdb_files = list(output_path.glob("*.pdb"))
        return pdb_files[0] if pdb_files else None


def rosetta_relax(pdb_file, output_pdb=None, nstruct=5):
    """
    Relax/refine structure using Rosetta relax application.
    This is simpler than full docking and useful for structure refinement.
    
    Parameters:
    - pdb_file: Path to input PDB file
    - output_pdb: Path to output PDB file (default: auto-generated)
    - nstruct: Number of relaxed structures to generate (default: 5)
    
    Returns:
    - Path to best relaxed structure, or None if relaxation fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available. Install Rosetta first.")
        return None
    
    relax_cmd = get_rosetta_command("relax")
    if not relax_cmd:
        print("⚠ Rosetta 'relax' command not found")
        print("   Available commands:", list(rosetta_commands.keys()))
        return None
    
    if output_pdb is None:
        output_pdb = Path(pdb_file).stem + "_relaxed.pdb"
    
    output_path = Path(output_pdb).parent
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("Rosetta Structure Relaxation")
    print("=" * 60)
    print(f"Input:  {pdb_file}")
    print(f"Output: {output_pdb}")
    print(f"Structures: {nstruct}")
    
    try:
        cmd = [
            relax_cmd,
            "-s", str(pdb_file),
            "-nstruct", str(nstruct),
            "-relax:fast",
            "-out:path:pdb", str(output_path),
            "-out:file:scorefile", str(output_path / "relax_scores.sc")
        ]
        
        print(f"\nRunning: {' '.join(cmd[:3])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            # Find best structure
            score_file = output_path / "relax_scores.sc"
            if score_file.exists():
                best_structure = _find_best_rosetta_structure(output_path)
                if best_structure:
                    print(f"\n✓ Relaxation completed!")
                    print(f"Best structure: {best_structure}")
                    return str(best_structure)
            
            print("\n✓ Relaxation completed (check output directory)")
            return str(output_path)
        else:
            print(f"\n⚠ Rosetta relax failed:")
            print(result.stderr[:500])
            return None
            
    except Exception as e:
        print(f"\n⚠ Error running Rosetta relax: {e}")
        return None


def rosetta_docking_protocol(receptor_pdb, ligand_pdb, output_dir="rosetta_docking_protocol",
                             nstruct=10, docking_method="local_refine"):
    """
    Use Rosetta docking_protocol for protein-protein/peptide-protein docking.
    This is more robust than rosetta_scripts for docking.
    
    Parameters:
    - receptor_pdb: Path to receptor PDB file
    - ligand_pdb: Path to ligand/peptide PDB file
    - output_dir: Directory to save results
    - nstruct: Number of structures to generate
    - docking_method: "local_refine" (default) or "perturb"
    
    Returns:
    - Path to best docked structure, or None if docking fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available. Install Rosetta first.")
        return None
    
    docking_protocol = get_rosetta_command("docking_protocol")
    if not docking_protocol:
        print("⚠ docking_protocol not found")
        print("   Falling back to rosetta_scripts...")
        return rosetta_peptide_docking(receptor_pdb, ligand_pdb, output_dir, nstruct)
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("Rosetta Docking Protocol")
    print("=" * 60)
    print(f"Receptor: {receptor_pdb}")
    print(f"Ligand:   {ligand_pdb}")
    print(f"Method:   {docking_method}")
    print(f"Structures: {nstruct}")
    
    try:
        cmd = [
            docking_protocol,
            "-s", str(receptor_pdb), str(ligand_pdb),
            "-nstruct", str(nstruct),
            "-docking", docking_method,
            "-out:path:pdb", str(output_path),
            "-out:file:scorefile", str(output_path / "docking_scores.sc")
        ]
        
        print(f"\nRunning: {' '.join(cmd[:5])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            best_structure = _find_best_rosetta_structure(output_path)
            if best_structure:
                print(f"\n✓ Docking completed!")
                print(f"Best structure: {best_structure}")
                return str(best_structure)
            else:
                print("\n✓ Docking completed (check output directory)")
                return str(output_path)
        else:
            print(f"\n⚠ Docking failed:")
            print(result.stderr[:500])
            return None
            
    except Exception as e:
        print(f"\n⚠ Error running docking_protocol: {e}")
        return None


def rosetta_fixbb(pdb_file, output_pdb=None, resfile=None):
    """
    Use Rosetta fixbb (fix backbone) to redesign side chains.
    Useful for structure optimization.
    
    Parameters:
    - pdb_file: Path to input PDB file
    - output_pdb: Path to output PDB file
    - resfile: Optional resfile for specific residue design
    
    Returns:
    - Path to output structure, or None if fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available")
        return None
    
    fixbb_cmd = get_rosetta_command("fixbb")
    if not fixbb_cmd:
        print("⚠ fixbb not found")
        return None
    
    if output_pdb is None:
        output_pdb = Path(pdb_file).stem + "_fixbb.pdb"
    
    output_path = Path(output_pdb).parent
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("Rosetta FixBB (Side Chain Design)")
    print("=" * 60)
    print(f"Input:  {pdb_file}")
    print(f"Output: {output_pdb}")
    
    try:
        cmd = [fixbb_cmd, "-s", str(pdb_file), "-out:path:pdb", str(output_path)]
        if resfile:
            cmd.extend(["-resfile", str(resfile)])
        
        print(f"\nRunning: {' '.join(cmd[:3])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"\n✓ FixBB completed!")
            return str(output_pdb)
        else:
            print(f"\n⚠ FixBB failed: {result.stderr[:300]}")
            return None
            
    except Exception as e:
        print(f"\n⚠ Error: {e}")
        return None


# Print Rosetta status and usage
print("\n" + "=" * 60)
print("Rosetta Functions Available")
print("=" * 60)

if ROSETTA_AVAILABLE:
    print("\n✓ Rosetta is ready!")
    print("\nAvailable functions:")
    print("1. rosetta_relax() - Structure relaxation/refinement")
    print("2. rosetta_peptide_docking() - Peptide-protein docking (rosetta_scripts)")
    print("3. rosetta_docking_protocol() - Docking using docking_protocol")
    print("4. rosetta_fixbb() - Side chain redesign")
    print("\nExample usage:")
    print("  # Relax structure")
    print("  relaxed = rosetta_relax('3fxi.pdb', nstruct=5)")
    print("  ")
    print("  # Dock peptide to receptor")
    print("  docked = rosetta_peptide_docking('3fxi.pdb', 'peptide.pdb', nstruct=10)")
else:
    print("\n⚠ Rosetta not installed")
    print("\nTo install Rosetta:")
    print("  Option 1: ./install_rosetta.sh (from GitHub)")
    print("  Option 2: conda install -c conda-forge rosetta")
    print("  Option 3: See INSTALL_MANUAL.md")
    print("\nNote: Rosetta is required for this workflow")

print("=" * 60)


In [ ]:
#relaxed = rosetta_relax('3fxi.pdb', nstruct=1)
#docked = rosetta_peptide_docking(relaxed, 'alphafold_predictions/ACDEFGHIKLMNPQRSTVWY.pdb', nstruct=1)

In [ ]:

# Dock peptide to receptor
# Use HPC if available (will auto-detect if HPC_ENABLED)
# To force HPC: use_hpc=True
# To force local: use_hpc=False
docked = rosetta_peptide_docking(
    '3fxi_0001.pdb', 
    'alphafold_predictions/ACDEFGHIKLMNPQRSTVWY_alphafold.pdb', 
    nstruct=1,
    use_hpc=None  # Auto-detect (uses HPC if available and not on HPC)
    # use_hpc=True  # Force HPC execution  
    # use_hpc=False  # Force local execution
)
